In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
import pandas as pd
from sklearn.datasets import load_iris
from pyspark.sql.functions import col, expr, rand

# Catalogo e Esquema do dataset Iris
catalogo_origem = "workspace"
esquema_origem = "default"
tabela_iris_raw = f"{catalogo_origem}.{esquema_origem}.iris_dataset"

# Lendo os dados brutos que já estão no ambiente
df_iris = spark.table(tabela_iris_raw)

# Garanta que a tabela possua a chave primária (id_flor) e as colunas ajustadas.


# Registra o DataFrame atual como uma View temporária no Spark
df_iris.createOrReplaceTempView("vw_iris_raw")

# Executa a query SQL para construir a chave primária
df_iris_sql = spark.sql("""
    SELECT 
        CONCAT('flor_', ROW_NUMBER() OVER (ORDER BY (SELECT NULL))) AS id_flor,
        sepal_length,
        sepal_width,
        petal_length,
        petal_width,
        species
    FROM vw_iris_raw
""")

# Sobrescreve o DataFrame original com o resultado do SQL
df_iris_new = df_iris_sql

display(df_iris_new.limit(5))


In [0]:
from databricks import feature_engineering

# Definindo o destino das nossas features
nome_tabela_features = f"{catalogo_origem}.{esquema_origem}.iris_features_table"

# Instanciando o cliente de engenharia Databricks
fe = feature_engineering.FeatureEngineeringClient()

# Isolando apenas os recursos (Features) e a chave primária que vão para a store
# Removemos a coluna target (species) para que ela não seja exposta na tabela de recursos
df_features_only = df_iris_new.select("id_flor", "sepal_length", "sepal_width", "petal_length", "petal_width")

print(f" Gravando e registrando a Feature Table em: {nome_tabela_features}...")

# Criando e populando a tabela de recursos de forma centralizada
fe.create_table(
    name=nome_tabela_features,
    primary_keys=["id_flor"],
    df=df_features_only,
    schema=df_features_only.schema,
    description="Tabela de recursos morfológicos das flores Iris integrada à Feature Store do Unity Catalog."
)
print("Features salvas e sincronizadas com sucesso! Você já consegue visualizá-las pelo Catalog Explorer.")

In [0]:
from pyspark.sql.functions import col

# 1. Carrega a tabela de recursos atual a partir da Feature Store para derivar a nova informação
df_existente = spark.table(nome_tabela_features)

print("Calculando a nova feature: razão morfológica da pétala (petal_ratio)...")

# Criando a nova variável baseada em colunas existentes (Razão = Comprimento / Largura)
# Mantemos obrigatoriamente a chave primária 'id_flor' no DataFrame resultante
df_nova_feature = df_existente.select(
    col("id_flor"),
    (col("petal_length") / col("petal_width")).alias("petal_ratio")
)

print(f"Atualizando a Feature Table no Unity Catalog: {nome_tabela_features}...")

# Gravando a nova característica usando o write_table
# O Databricks faz o merge inteligente baseado na chave primária ['id_flor']
fe.write_table(
    name=nome_tabela_features,
    df=df_nova_feature,
    mode="merge" # Realiza o update das linhas combinando as chaves primárias
)

print("Nova feature 'petal_ratio' adicionada e integrada com sucesso!")

# Visualizando o novo esquema atualizado
display(spark.table(nome_tabela_features).limit(5))

In [0]:
%sql
ALTER TABLE workspace.default.iris_features_table SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5');
ALTER TABLE workspace.default.iris_features_table DROP COLUMNS (petal_ratio);

In [0]:
%sql
SELECT * FROM workspace.default.iris_features_table

In [0]:
%sql
DESCRIBE HISTORY workspace.default.iris_features_table